In [ ]:
from typing import TypedDict,Annotated
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage,BaseMessage
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
import operator
from dotenv import load_dotenv

load_dotenv()

In [ ]:
class MessageState(TypedDict):
    messages: Annotated[list["BaseMessage"] , operator.add]

In [ ]:
llm = ChatGroq(model="llama-3.1-8b-instant")

# messages = [
#     SystemMessage(content="You are a helpful assistant.Answer in one sentence."),
#     HumanMessage(content="What is the captial of france"),
# ]

# response = llm.invoke(messages)
# print(response.content)

In [ ]:
SYSTEM = SystemMessage(content="You are a helpful assistant.You remember everything in this")

def chatbot_node(state:MessageState)-> dict:
    all_messages = [SYSTEM] + state["messages"]
    response = llm.invoke(all_messages)
    print(f"Chabot History length is {len(state["messages"])}")
    return {"messages":[response]}

In [ ]:
builder = StateGraph(MessageState)
builder.add_node("chatbot_node",chatbot_node)

builder.add_edge(START,"chatbot_node")
builder.add_edge("chatbot_node",END)
0
graph = builder.compile();

In [ ]:
print("----------------------------------Multi Turn Chatbot------"
"-------------------------------")
history:list[BaseMessage] = []
user_text_list = [
    "Hi,My name is Balaji and I live in Pune.",
    "I am learning LangGraph framework.",
    "What is my name,where do i leave and what i am learning?",
    "Summarise our entire conversation."
]

#for user_text in user_text_list:
#input()

while True:
    user_text = input().lower()
    if user_text == "exit" or user_text == "quit":
        break

    history.append(HumanMessage(content=user_text))
    response = graph.invoke({"messages" : history})
    #print(response)
    history.append(response["messages"][-1])
    
    print(f"User Input:  {user_text}")
    print(f"Chatbot Response: {(response["messages"][-1]).content}")
    print("-"*90)